<a href="https://colab.research.google.com/github/karquiza/bikeshare-toronto-demand/blob/main/notebooks/M4_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAI 103 Group 11 Assignment 2 - Milestone 4: Baseline Modeling, Evaluation & Data Generation Strategy

## Bike Share Station Demand for Growth Planning

### Members
- TODO: Add members here when ready to submit

## Part 1: Experimental Design & Data Preparation

### 1.1 Modeling Objective

**Problem Formulation:** Can we predict the hourly departures and arrivals at bike stations and classify whether the demand for a station is Low, Medium, or High, so that Bike Share Toronto can plan where to add new docks or stations?

**Target Variables:**

1. **Regression:** Departures per station per hour, Arrivals per station per hour during the peak months (May to October).
- For each month, a station is considered as open/operating if there is at least 1 trip starting at that station.
- For open/operating stations, hours with zero trips (ex. typical of night time hours) are included in the model.
- In Milestone 3, we set the following flags in the cleaned data to mark valid or invalid trips:
    - `flag_short`: Trip duration is under 60 seconds
    - `flag_long`: Trip duration is over 6 hours
    - `flag_no_end`: The trip has no `End_Station_Id`
    - `is_valid`: True when none of the flags is set, False if at least one is set
- (a) Departures - Valid trips (rows) have no `flag_short` or `flag_long` set. A missing end station doesn't matter.
- (b) Arrivals - Valid trips (rows) have none of the flags (`flag_short`, `flag_long`, `flag_no_end`) set.
    - Note: We said in Milestone 3 that we would no longer include Arrivals as a target variable due to the missing `End_Station_Id`s. However, after discussion, we have decided that the small percentage of missing `End_Station_Id`s should not affect the model too much, and this target variable is relevant in determining how busy a station is.

2. **Classification:** Predict the demand of stations during May-October 2025 as Low, Medium, or High, based on the training data from the same months in 2023 and 2024.
- One row is one station in one month. A station-month exists if the station had at least 1 valid departure in that month.
- Demand rate for a station is calculated by the average valid departures per operating day for this version.
    - An operating day is all the days in an operating month (if there is at least 1 valid departure in that month, all days in that month will be considered operating days). This is the same rule as we used for the Regression task, and we keep it here for consistency.
    - Days with no trips in an operating month will have a zero demand.
    - Valid departures use the same `is_valid` flag as in Milestone 3. Arrivals are not included in this rate.
    - Note: There are changes from Milestone 3, namely changing the definition to use operating days instead of the active days, because using active days only hides low-demand stations that may have few active days.
- The thresholds for the classification are the following:
    - **TODO: Add code to recompute target variables here. Low=lowest 1/3, Medium=middle 1/3, High=highest 1/3 cutoffs based on the training data only.**
- Stations that first appear in 2025 have no 2023 to 2024 rate and are not in the classification data. They are used only in the business check that compares 2024 High stations with the new 2025 stations.
    

**Features:**

Each ML task has a different set of features.

1. **Regression**
- One row per station per hour
- (a) Features from date/timestamp:

     - `hour_of_day` - categorical values: hr_0-hr_23, rush hour will have different behaviour
     - `day_of_week` - categorical values: Monday-Sunday, riding patterns are different on weekdays and weekends
     - `weekend_or_holiday` - flag (1 if weekend or holiday, 0 otherwise)
     - `year` - numerical values: 2023, 2024, 2025
     - `month` - categorical values: May-Oct

- (b) Features from weather (TBD)

    - `prec_mm_hr` - numerical value, precipitation/rain in that hour
    - `mean_temp_c_hr` - numerical value, temperature in that hour
    - `prec_mm_prev_hr` - numerical value, precipitation/rain in the previous hour (since people check weather before deciding to ride)

- (c) Features from station:

    - `station_id` - categorical
    - `new_station` - flag (1 if first month that the station is seen, 0 otherwise). We are doing regression on May to October months only, but we have data from January 2023. We will check for the first occurrence of a station since the very beginning of our data (January 2023) in order to set this flag. (Note that this information is in Milestone 3, since we cleaned the data using the entire dataset)


2. **Classification**

- One row per station per month (May to October)
- (a) Features from month's activity:
     - `monthly_volume` - numerical value, valid departures from the station in that month.
      - `was_flagged_new` - flag (1 if that month is the first month the station appears in the data, 0 otherwise). `is_new_start_station` flag from Milestone 3 on monthly level.

- (b) Features from station history (Milestone 3 station months)
    - `first_year` - numerical value, the year the station first appears in the data.
    - `first_month` - numerical value, the month the station first appears in the data.

### 1.2 Evaluation Criteria aligned with the Problem

Our training-validation-test split has been changed since Milestone 3:
- Training: May-October 2023 + May 2024
- Validation: June-October 2024
- Test: May-October 2025

We will score each model using the Validation set to help choose the best settings (hyperparameter tuning) and models. Once a model and/or its hyperparameter settings have been chosen, we will rerun the training on the Training+Validation set and evaluate this final model on the Test set once at the end.

1. **Regression** (departures per hour, arrivals per hour)
- Performance metrics: MAE, RMSE for departures and arrivals separately, since they are separate models.
- Baseline to compare against: average departures/arrivals for the same station, hour of day, and day of week in the training months.
- Success criteria: MAE on Test set should be lower than the baseline for both departures and arrivals.

2. **Classification**
- Performance metrics: TODO based on the Classification notebooks

3. **Business check**
- Compare the stations classified as High in 2024 with the stations that first appear in 2025, to see whether Bike Share Toronto's expansion followed the high-demand stations. This is to check whether our project's outputs would be useful for growth planning, according to the problem formulation we were trying to solve.

## Part 2: Data Generation / Augmentation (If Applicable)

## Part 3: Final Model Development & Optimization

## Part 4: Model Evaluation & Analytical Depth

## Part 5: Analytical Reflection (1 Page Required-Very important)